In [1]:
!pip install streamlit pypdf faiss-cpu pydantic \
  langchain langchain-core langchain-community langchain-text-splitters \
  langchain-huggingface sentence-transformers
!pip install -q langchain-community faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 70.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 86.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 84.9 MB/s eta 0:00:00:00:010:01


# **Calling The Model From Hugging Face & Testing The Model**

In [2]:
from transformers import pipeline
import torch

pipe = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct"
                ,torch_dtype=torch.float16,device_map="auto" , 
               max_new_tokens=2000, do_sample = True, temperature=0.4,
                return_full_text=False)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [3]:
messages = [
    {"role": "user", "content": "Who are you?"},
]
OUTPUT = pipe(messages )

[transformers] Both `max_new_tokens` (=2000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [4]:
OUTPUT

[{'generated_text': 'I am Qwen, created by Alibaba Cloud. I am here to assist you with various tasks and answer your questions to the best of my abilities. How can I help you today?'}]

# **Creating LLM Object** (Converting The Model Into ChatModel)

In [5]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))

In [6]:
print(llm.invoke("Hello").content)

[transformers] Both `max_new_tokens` (=2000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hello! How can I assist you today? Feel free to ask any questions or let me know if you need information on a particular topic.


# **Reading The Dataset**

In [7]:
!pip install -q pypdf langchain-text-splitters

In [8]:
from pypdf import PdfReader

def read_pdf(path):
    return "\n".join(p.extract_text() or "" for p in PdfReader(path).pages)

In [9]:
text = read_pdf("/kaggle/input/datasets/engmohammadelprince/dataset-trainning/FLUTTER.pdf")

# **Creating Chunks Of Main Text**

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_text(text)
print(len(chunks))
print(chunks[0][:300])

2
FLUTTER 
(Mobile Application)
Mohammad Prince Abdelrahman
Application Design
Material 
App
Home
(Scaffold)
Body App Bar
Title Actions
Title
FLUTTER WIDGETS


# **RAG For Summarization Process**

In [24]:
from langchain_huggingface import HuggingFaceEmbeddings

emb = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [25]:
v = emb.embed_query("What is Flutter?")
print(len(v), v[:5])

384 [-0.2429949939250946, -0.18973815441131592, -0.14490468800067902, -0.30085310339927673, 0.03764181211590767]


In [26]:
from langchain_community.vectorstores import FAISS

store = FAISS.from_texts(chunks, emb)
print("done")

done


In [27]:
question = "What is Flutter ?"

docs = store.similarity_search(question, k=3)

for i, d in enumerate(docs, 1):
    print(f"--- Result {i} ---")
    print(d.page_content[:300])
    print()

--- Result 1 ---
FLUTTER 
(Mobile Application)
Mohammad Prince Abdelrahman
Application Design
Material 
App
Home
(Scaffold)
Body App Bar
Title Actions
Title
FLUTTER WIDGETS

--- Result 2 ---
Container
❑ Properties :-
• width (width : double.infinity ) 
• height (height : double.infinity ) 
• color (color : Colors.red )
• margin (margin : EdgeInsets.all (num))
• padding (padding : EdgeInsets.all (num))
• child (child : Text(“ bla bla bla ”))
• alignment (alignment : Alignment.center)
wid



In [15]:
results = store.similarity_search_with_score(question, k=3)

for d, score in results:
    print(round(float(score), 3), "→", d.page_content[:100].replace("\n", " "))

27.367 → FLUTTER  (Mobile Application) Mohammad Prince Abdelrahman Application Design Material  App Home (Sca
36.829 → Container ❑ Properties :- • width (width : double.infinity )  • height (height : double.infinity )  


# **Creating Chain** (Prompt-LLM-OutputParser)

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

qa_chain = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below. "
    "If the answer is not in the context, say: I could not find this in the lecture.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
) | llm | StrOutputParser()

In [18]:
def ask(question, k=4):
    docs = store.similarity_search(question, k=k)
    context = "\n\n".join(d.page_content for d in docs)
    answer = qa_chain.invoke({"context": context, "question": question})
    return answer, docs

In [28]:
answer, docs = ask("What is an margin?")
print(answer)

[transformers] Both `max_new_tokens` (=2000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Margin in Flutter refers to the space surrounding the content inside a container. It specifies the distance between the content and the edges of the container. The `margin` property of the Container widget can be set using `EdgeInsets.all(num)` where `num` is the size of the margin in pixels.


In [22]:
for d in docs:
    print("-", d.page_content[:150].replace("\n", " "))

- FLUTTER  (Mobile Application) Mohammad Prince Abdelrahman Application Design Material  App Home (Scaffold) Body App Bar Title Actions Title FLUTTER WI
- Container ❑ Properties :- • width (width : double.infinity )  • height (height : double.infinity )  • color (color : Colors.red ) • margin (margin : E


In [23]:
print(len(text))

934


# **Dividing The Dataset Into Other Small Datasets**

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=3500, chunk_overlap=200)
parts = splitter.split_text(text)

In [30]:
print(len(parts))
print([len(e) for e in parts])

1
[920]


In [32]:
summ_chain = ChatPromptTemplate.from_template(
    "Summarize the following text as 8 clear bullet points grouped by topic. "
    "Use ONLY the information in the text. Write in {language}.\n\n{text}"
) | llm | StrOutputParser()

In [33]:
print(summ_chain.invoke({"text": parts[0], "language": "English"}))

[transformers] Both `max_new_tokens` (=2000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


- **Flutter Mobile Application**
  - **Design Topic: Material App**
    - **Home Scaffold**
      - **App Bar Body**
        - **Title Actions**
          - **Title Text**
            - **Properties of Container Widget**
              - **Width and Height**
                - **Double Infinity**
              - **Color**
                - **Red Color**
              - **Margin**
                - **EdgeInsets All**
              - **Padding**
                - **EdgeInsets All**
              - **Child Element**
                - **Text Element**
              - **Alignment**
                - **Center Alignment**
            - **Container Widget Properties**
              - **Decoration**
                - **BoxDecoration Properties**
                  - **Color**
                    - **Adjustable Color**
                  - **Shape**
                    - **Circle Shape**
                  - **Border Radius**
                    - **Radius Circular**
                  - **Border**
  

In [34]:
partial = summ_chain.batch([{"text": p, "language": "English"} for p in parts])

[transformers] Both `max_new_tokens` (=2000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [35]:
merged = "\n".join(partial)
print(len(merged))

666


In [36]:
final_summary = summ_chain.invoke({"text": merged[:7000], "language": "Engish"})
print(final_summary)

[transformers] Both `max_new_tokens` (=2000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


- **Flutter Mobile Application**
  - **Design Elements**
    - Material Design
    - App Scaffold
      - Home Screen
        - AppBar Body
          - Title Actions
            - Title Element
  - **Flutter Widgets**
    - Container Widget
      - Properties
        - Width and Height
        - Color
        - Margin
        - Padding
        - Child Element
        - Alignment
      - Example Usage
        - Text Content
    - Container Widget Decorations
      - Properties
        - Color
        - Shape
        - Border Radius
        - Border
        - Box Shadow


# **Quiz Creation**

In [37]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [39]:

quiz_prompt = PromptTemplate.from_template(
    "Create {n} {level} multiple-choice questions based ONLY on the text. "
    "Write in {language}.\n{format}\n\nText:\n{text}"
)

quiz_chain = quiz_prompt | llm | JsonOutputParser()

In [40]:
FORMAT = '''Return ONLY a JSON list, with no extra text. Each item must look like:
{"topic": "...", "question": "...", "options": ["...", "...", "...", "..."], "answer": "one of the options, copied exactly", "explanation": "one sentence"}'''

quiz = quiz_chain.invoke({"n": 3, "level": "medium", "language": "English",
                          "format": FORMAT, "text": final_summary[:6000]})
print(type(quiz))       
print(quiz[0])         
print(quiz[0]["question"])
print(quiz[0]["options"])

[transformers] Both `max_new_tokens` (=2000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<class 'list'>
{'topic': 'Material Design', 'question': 'Which design element is associated with Flutter Mobile Applications?', 'options': ['Material Design', 'App Scaffold', 'Home Screen', 'AppBar Body'], 'answer': 'Material Design', 'explanation': 'The question directly asks about design elements related to Flutter, and Material Design is one of the design elements mentioned in the text.'}
Which design element is associated with Flutter Mobile Applications?
['Material Design', 'App Scaffold', 'Home Screen', 'AppBar Body']


In [42]:
def is_valid(q):
    keys = ("topic", "question", "options", "answer", "explanation")
    return (isinstance(q, dict)
            and all(k in q for k in keys)
            and isinstance(q["options"], list)
            and len(q["options"]) == 4
            and q["answer"].strip() in [o.strip() for o in q["options"]])

good = [q for q in quiz if is_valid(q)]
print(len(good), "of", len(quiz), " are valid")

3 of 3  are valid


In [43]:
questions = good
answers = []

for i, q in enumerate(questions, 1):
    print(f"\n{i}. {q['question']}")
    for j, o in enumerate(q["options"], 1):
        print(f"   {j}) {o}")
    choice = int(input("your answer (1-4): ")) - 1
    answers.append(q["options"][choice])


1. Which design element is associated with Flutter Mobile Applications?
   1) Material Design
   2) App Scaffold
   3) Home Screen
   4) AppBar Body


your answer (1-4):  4



2. Which property can be used to set the color of a Container widget?
   1) Width and Height
   2) Color
   3) Margin
   4) Padding


your answer (1-4):  2



3. What does the 'Container' widget's 'Alignment' property do?
   1) Sets the width and height
   2) Sets the color of the container
   3) Determines the position within the parent layout
   4) Creates a box shadow


your answer (1-4):  3


In [44]:
wrong = [(q, a) for q, a in zip(questions, answers) if a != q["answer"].strip()]

In [45]:
print(wrong)

[({'topic': 'Material Design', 'question': 'Which design element is associated with Flutter Mobile Applications?', 'options': ['Material Design', 'App Scaffold', 'Home Screen', 'AppBar Body'], 'answer': 'Material Design', 'explanation': 'The question directly asks about design elements related to Flutter, and Material Design is one of the design elements mentioned in the text.'}, 'AppBar Body')]


In [46]:
wrong_txt = "\n".join(
    f"- [{q['topic']}] {q['question']} | student answered: {a} | correct: {q['answer']}"
    for q, a in wrong
) or "none"
print(wrong_txt)

- [Material Design] Which design element is associated with Flutter Mobile Applications? | student answered: AppBar Body | correct: Material Design


In [47]:
score = sum(a == q["answer"].strip() for q, a in zip(questions, answers))


In [48]:
pct = score / len(questions) * 100
print(f"{score}/{len(questions)}  ({pct:.0f}%)")

2/3  (67%)


# **Evaluation Part**

In [50]:
eval_chain = ChatPromptTemplate.from_template(
    "You are a supportive tutor. A student scored {score}/{total} on a quiz.\n"
    "Wrong answers:\n{wrong}\n\n"
    "Write a short evaluation: overall performance, strengths, weak topics, "
    "and 3 concrete study tips. Write in {language}."
) | llm | StrOutputParser()

In [51]:
print(eval_chain.invoke({
    "score": score,
    "total": len(questions),
    "wrong": wrong_txt,
    "language": "English",
}))

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=2000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Evaluation of the Student's Performance

The student scored 2 out of 3 on the quiz, indicating that they have some understanding but also significant gaps in their knowledge. The score suggests that the student has a basic grasp of one concept (the incorrect answer was not related to the topic), but struggles with another fundamental aspect of Flutter mobile applications.

### Strengths

- **Basic Understanding**: The student demonstrated an ability to recall at least one concept correctly, which shows a foundational understanding of the subject matter.
  
### Weak Topics

- **Flutter Concepts**: The student struggled with the concept of Material Design, specifically identifying the correct design element associated with Flutter mobile applications. This indicates a weakness in comprehending specific aspects of Flutter development.

### Study Tips

1. **Review Flutter Basics**: Start by revisiting the basics of Flutter, particularly focusing on the core concepts such as Material De

# **Streamlit**

In [52]:
!pip install -q streamlit pyngrok

In [2]:
!pkill -f streamlit

In [70]:
import gc, torch
del llm, pipe
gc.collect()
torch.cuda.empty_cache()

NameError: name 'llm' is not defined

t=2026-10-06T20:23:38+0000 lvl=warn msg="failed to open private leg" id=740379c60e96 privaddr=localhost:8501 err="dial tcp [::1]:8501: connect: connection refused"
t=2026-10-06T20:23:38+0000 lvl=warn msg="failed to open private leg" id=a5c925254ed7 privaddr=localhost:8501 err="dial tcp [::1]:8501: connect: connection refused"
t=2026-10-06T20:23:42+0000 lvl=warn msg="failed to open private leg" id=81f0df0bd621 privaddr=localhost:8501 err="dial tcp [::1]:8501: connect: connection refused"
t=2026-10-06T20:23:42+0000 lvl=warn msg="failed to open private leg" id=46b369ac01e9 privaddr=localhost:8501 err="dial tcp [::1]:8501: connect: connection refused"


In [84]:
%%writefile app.py
import torch
import streamlit as st
from pypdf import PdfReader
from langchain_core.prompts import PromptTemplate , ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser , StrOutputParser
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace , HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


st.set_page_config(page_title="Study Chat Bot", page_icon="📖", layout="centered")

st.markdown("""
<style>
#MainMenu, footer {visibility: hidden;}
.block-container {padding-top: 4rem; max-width: 850px;}
.stButton > button {border-radius: 10px; padding: 0.5rem 1.2rem;}
</style>
""", unsafe_allow_html=True)

@st.cache_resource(show_spinner="Loading model (first time takes a few minutes)...")
def get_llm():
    pipe = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct",
                    torch_dtype=torch.float16, device_map="auto",
                    max_new_tokens=2000, do_sample=True, temperature=0.4,
                    return_full_text=False)
    return ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))


def read_pdf(file):
    return "\n".join(p.extract_text() or "" for p in PdfReader(file).pages)


def summarize(llm, text, language="English", max_parts=3):
    chain = ChatPromptTemplate.from_template(
        "Summarize the following text as clear bullet points grouped by topic. "
        "Use ONLY the information in the text. Write in {language}.\n\n{text}"
    ) | llm | StrOutputParser()
    parts = RecursiveCharacterTextSplitter(chunk_size=3500, chunk_overlap=200).split_text(text)[:max_parts]
    partial = chain.batch([{"text": p, "language": language} for p in parts])
    if len(partial) == 1:
        return partial[0]
    return chain.invoke({"text": "\n".join(partial)[:7000], "language": language})


@st.cache_resource(show_spinner="Building search index...")
def build_store(text):
    chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_text(text)
    emb = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    return FAISS.from_texts(chunks, emb)
    
def ask(llm, store, question, k=4):
    docs = store.similarity_search(question, k=k)
    context = "\n\n".join(d.page_content for d in docs)
    chain = ChatPromptTemplate.from_template(
        "Answer the question using ONLY the context below. "
        "If the answer is not in the context, say: I could not find this in the lecture.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}"
    ) | llm | StrOutputParser()
    return chain.invoke({"context": context, "question": question}), docs

def is_valid(q):
    keys = ("topic", "question", "options", "answer", "explanation")
    return (isinstance(q, dict)
            and all(k in q for k in keys)
            and isinstance(q["options"], list)
            and len(q["options"]) == 4
            and isinstance(q["answer"], str)
            and q["answer"].strip() in [str(o).strip() for o in q["options"]])


def generate_quiz(llm, summary, n=5, level="medium", language="English", retries=2):
    chain = PromptTemplate.from_template(
        "Create {n} {level} multiple-choice questions based ONLY on the text. "
        "Do not reveal the answer inside the question. Write in {language}.\n{format}\n\nText:\n{text}"
    ) | llm | JsonOutputParser()

    for _ in range(retries + 1):
        try:
            quiz = chain.invoke({"n": n, "level": level, "language": language,
                                 "format": FORMAT, "text": summary[:6000]})
            if isinstance(quiz, dict):
                quiz = quiz.get("questions", [])
            good = [q for q in quiz if is_valid(q)]
            if good:
                for q in good:
                    q["options"] = [str(o).strip() for o in q["options"]]
                    q["answer"] = q["answer"].strip()
                return good[:n]
        except Exception:
            pass
    return []

def show_message(m):
    with st.chat_message(m["role"]):
        st.markdown(m["content"])
        if m.get("sources"):
            with st.expander("📚 Sources"):
                for s in m["sources"]:
                    st.caption(s[:300])
                    
ss = st.session_state
ss.setdefault("stage", "upload")
ss.setdefault("chat", [])

with st.sidebar:
    st.header("📖 Study Chat Bot")
    st.caption("Summarize a lecture, chat with it, then test yourself.")
    if ss.stage != "upload":
        if st.button("📄 New lecture", key="new_lecture"):
            for k in ["summary", "text", "questions", "answers", "qi"]:
                ss.pop(k, None)
            ss.chat = []
            ss.stage = "upload"
            st.rerun()

FORMAT = '''Return ONLY a JSON list, with no extra text. Each item must look like:
{"topic": "...", "question": "...", "options": ["...", "...", "...", "..."], "answer": "one of the options, copied exactly", "explanation": "one sentence"}'''



if ss.stage == "upload":
    st.title("📖 Study Chat Bot")
    st.caption("Upload a lecture, get a summary, ask questions, then test yourself.")
    pdf = st.file_uploader("Upload your lecture (PDF)", type="pdf")
    if st.button("Summarize", type="primary", key="summarize_btn") and pdf:
        text = read_pdf(pdf)
        if len(text.strip()) < 200:
            st.warning("The PDF has almost no text (maybe scanned images).")
        else:
            llm = get_llm()
            with st.spinner("Summarizing..."):
                ss.summary = summarize(llm, text)
                ss.text = text
            ss.stage = "summary"
            st.rerun()

elif ss.stage == "summary":
    st.title("Study Bot")
    tab_sum, tab_chat, tab_quiz = st.tabs(["Summary", "Chat", "Quiz"])

    with tab_sum:
        st.markdown(ss.summary)
        
    with tab_chat:
        if not ss.chat:
            st.info("Ask anything about the lecture. Answers come only from your file.")
        for m in ss.chat:
            show_message(m)
        if question := st.chat_input("Ask about the lecture..."):
            show_message({"role": "user", "content": question})
            with st.chat_message("assistant"):
                with st.spinner("Thinking..."):
                    answer, docs = ask(get_llm(), build_store(ss.text), question)
            ss.chat += [
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer,
                 "sources": [d.page_content for d in docs]},
            ]
            st.rerun()
    
    with tab_quiz:
        n = st.slider("Number of questions", 3, 10, 5)
        if st.button("Start quiz"):
            llm = get_llm()
            with st.spinner("Generating quiz..."):
                ss.questions = generate_quiz(llm, ss.summary, n)
            if not ss.questions:
                st.error("Could not generate a valid quiz. Try again.")
            else:
                ss.stage = "quiz"
                ss.qi = 0
                ss.answers = []
                st.rerun()

elif ss.stage == "quiz":
    questions = ss.questions
    q = questions[ss.qi]
    st.progress(ss.qi / len(questions), text=f"Question {ss.qi + 1} of {len(questions)}")
    with st.container(border=True):
             st.subheader(q["question"])
             choice = st.radio("Choose one", q["options"], index=None,
                               key=f"r{ss.qi}", label_visibility="collapsed")
    if st.button("Confirm ✅", type="primary", key="confirm_btn"):
        if choice is None:
            st.warning("Choose an answer first")
        else:
            ss.answers.append(choice)
            ss.qi += 1
            if ss.qi == len(questions):
                ss.stage = "result"
            st.rerun()

elif ss.stage == "result":
    questions = ss.questions
    score = sum(a == q["answer"] for q, a in zip(questions, ss.answers))
    pct = score / len(questions)

    c1, c2, c3 = st.columns(3)
    c1.metric("Score", f"{score}/{len(questions)}")
    c2.metric("Percent", f"{pct:.0%}")
    c3.metric("Grade", "🏆 Excellent" if pct >= .9 else "👏 Very good" if pct >= .75
              else "👍 Good" if pct >= .6 else "📚 Review")
    st.progress(pct)
    if pct >= .8:
        st.balloons()

    for i, (q, a) in enumerate(zip(questions, ss.answers), 1):
        ok = a == q["answer"]
        with st.expander(f"{'✅' if ok else '❌'}  {i}. {q['question']}"):
            st.write(f"**Your answer:** {a}")
            if not ok:
                st.write(f"**Correct answer:** {q['answer']}")
            st.caption(q["explanation"])

    if st.button("⬅️ Back to lecture", key="back_btn"):
        ss.stage = "summary"
        st.rerun()

Overwriting app.py


In [1]:
!pkill -f streamlit
!pkill -x ngrok

In [86]:
import subprocess, time
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

subprocess.run(["pkill", "-f", "streamlit"])

ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))

for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)

print(ngrok.connect(8501).public_url)

2026-10-06 20:49:00.745 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.55.200.198:8501

https://tackle-bounce-ascertain.ngrok-free.dev


/kaggle/working/app.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 9